# Understand Datasets

Edge: relationships between two products

Nodes: infomations for main products and their property

temporal data: daily production/ unit weight

# Edges

-- Edges Index

Edges(Plant):

Plant	node1	node2
1901	ATWWP001K24P	ATN01K24P
1903	AT5X5K	ATN01K24P


Edges(Product Group):

Edges(Product Sub-Group):

Edges(Storage Location):


-- Edges

Edges(Plant):

Edges(Product Group):

Edges(Product Sub-Group):

Edges(Storage Location):

# Nodes

# Temporal Data

Data audit -- Edges

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "config.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find config.py. Start Jupyter from the repository or its Notebooks folder."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from functools import reduce

import pandas as pd

from config import (
    DAILY_FLOW_FILE,
    DELIVERY_FILE,
    FACTORY_FILE,
    NODES_FILE,
    PRODUCTION_FILE,
    SALES_FILE,
    check_raw_data,
    ensure_output_directories,
)


In [ ]:
check_raw_data()
products_df = pd.read_csv(NODES_FILE)
products_df.head()


In [ ]:
sales_df = pd.read_csv(SALES_FILE)
production_df = pd.read_csv(PRODUCTION_FILE)
delivery_df = pd.read_csv(DELIVERY_FILE)
factory_df = pd.read_csv(FACTORY_FILE)


In [ ]:
# short table to long table

long_tables = []

sales_df["Date"] = pd.to_datetime(sales_df["Date"])
long_df_sales = sales_df.melt(
    id_vars="Date",
    var_name="raw_product_id",
    value_name="quantity"
)

long_df_sales.rename(columns = {"quantity":"sales_units", "raw_product_id":"product_id"}, inplace = True)


production_df["Date"] = pd.to_datetime(production_df["Date"])
long_df_production = production_df.melt(
    id_vars="Date",
    var_name="raw_product_id",
    value_name="quantity"
)

long_df_production.rename(columns = {"quantity":"production_units", "raw_product_id":"product_id"}, inplace = True)


delivery_df["Date"] = pd.to_datetime(delivery_df["Date"])
long_df_delivery = delivery_df.melt(
    id_vars="Date",
    var_name="raw_product_id",
    value_name="quantity"
)

long_df_delivery.rename(columns = {"quantity":"delivery_units", "raw_product_id":"product_id"}, inplace = True)

factory_df["Date"] = pd.to_datetime(factory_df["Date"])
long_df_factory = factory_df.melt(
    id_vars="Date",
    var_name="raw_product_id",
    value_name="quantity"
)

long_df_factory.rename(columns = {"quantity":"factory_units", "raw_product_id":"product_id"}, inplace = True)

long_tables = [long_df_sales, long_df_production, long_df_delivery, long_df_factory]

merged = reduce(
    lambda left, right: left.merge(right, on=["Date", "product_id"], how="outer"),
    long_tables
)

merged['product_id'] = merged['product_id'].replace({"POP001L12P.1" : "POP001L12P"})


In [ ]:
merged_groups = merged.merge(products_df[['Node', 'Group', 'Sub-Group']], left_on="product_id", right_on="Node", how="left").drop(columns=["Node"])

In [ ]:
flow_columns = [
    "Date",
    "product_id",
    "Group",
    "Sub-Group",
    "sales_units",
    "production_units",
    "delivery_units",
    "factory_units",
]
merged_groups = merged_groups[flow_columns]
merged_groups.head()


In [ ]:
daily_flow = merged_groups.groupby(
    ['Date', 'product_id', 'Group', 'Sub-Group'],
    as_index=False
).sum()

In [ ]:

print("Rows:", len(daily_flow))
print("Products:", daily_flow["product_id"].nunique())
print("Start date:", daily_flow["Date"].min())
print("End date:", daily_flow["Date"].max())
print("Duplicate keys:",
      daily_flow.duplicated(["Date", "product_id"]).sum())
print("Missing values:")
print(daily_flow.isna().sum())

In [ ]:
missing_products_id = daily_flow[daily_flow['Group'].isna()]['product_id'].unique()
print("Missing products_id:", missing_products_id)


In [ ]:
ensure_output_directories()
daily_flow.to_csv(DAILY_FLOW_FILE, index=False)
print(f"Saved: {DAILY_FLOW_FILE}")
